# Practical PyTorch · Phase II, Part 2 — Companion Notebook

This goes with **"The Five-Line Training Loop That Makes PyTorch Learn"**. Run the cells top to bottom (Shift+Enter).

Everything here trains in under a second on a plain CPU — no GPU needed, nothing to install. Colab ships with PyTorch already.

## 1. A tiny model and some toy data

We'll give the model the easiest possible job: learn the rule `y = 2x + 1`. We never tell it that rule — we just hand it `(x, y)` pairs that follow it and let it recover the slope (`2`) and the intercept (`1`) on its own.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)  # so your numbers match the post

# Toy data: y = 2x + 1, with a little noise so it's not too easy.
x = torch.linspace(-1, 1, 100).unsqueeze(1)   # shape [100, 1]
y = 2 * x + 1 + 0.1 * torch.randn_like(x)     # shape [100, 1]

print("x shape:", x.shape)
print("y shape:", y.shape)

# The smallest real model there is: learns one slope and one bias.
model = nn.Linear(1, 1)
print("starting weight:", model.weight.item())
print("starting bias:  ", model.bias.item())

The starting weight and bias are random — right now the model is confidently wrong. Our job is to fix that.

## 2. The loss function and the optimizer

The **loss function** scores how wrong the model is. The **optimizer** knows how to adjust the model's numbers to lower that score.

- `nn.MSELoss` — mean squared error, the standard choice when predicting a continuous number.
- `torch.optim.SGD` — plain gradient descent. We pass it `model.parameters()` (the numbers it's allowed to change) and a learning rate.

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

## 3. The loop, one line at a time

The canonical PyTorch training loop. Five steps, repeated for a number of **epochs** (one epoch = one pass over the data):

1. **forward** — `model(x)` makes guesses
2. **loss** — `loss_fn(preds, y)` scores how wrong
3. **zero_grad** — clear last round's gradients (they accumulate otherwise!)
4. **backward** — autograd computes the nudge for every parameter
5. **step** — the optimizer applies the nudge

In [ ]:
for epoch in range(100):
    preds = model(x)              # 1. forward:   guess
    loss = loss_fn(preds, y)      # 2. loss:      how wrong?

    optimizer.zero_grad()         # 3. clear old gradients
    loss.backward()               # 4. backward:  work out the nudge
    optimizer.step()              # 5. step:      apply the nudge

    if epoch % 10 == 0:
        print(f"epoch {epoch:3d}  loss {loss.item():.4f}")

That falling number **is** learning — there's no other magic.

## 4. Did it recover the rule?

Because the model only holds two numbers, we can check whether it found the slope (`2`) and intercept (`1`) we never told it about.

In [ ]:
print("learned weight:", round(model.weight.item(), 3), "(target ~2.0)")
print("learned bias:  ", round(model.bias.item(), 3), "(target ~1.0)")

Not exact — we added noise to the data — but the model recovered a rule it was never told. That's the whole point.

When you only want predictions (not learning), wrap the call in `torch.no_grad()` so autograd doesn't waste effort tracking it:

In [ ]:
model.eval()  # good habit: switch to evaluation mode before predicting
with torch.no_grad():
    test_x = torch.tensor([[3.0]])     # rule says y = 2*3 + 1 = 7
    print("prediction for x=3:", model(test_x).item())

## 5. See the `zero_grad` gotcha for yourself

Gradients **accumulate** — each `.backward()` adds to whatever is already stored. Here's what happens if you forget to zero them: the gradient keeps growing instead of being recomputed fresh each time.

In [ ]:
demo = nn.Linear(1, 1)
demo_loss_fn = nn.MSELoss()

print("WITHOUT zero_grad — gradient keeps piling up:")
for step in range(3):
    loss = demo_loss_fn(demo(x), y)
    loss.backward()  # no zero_grad() before this
    print(f"  step {step}: weight.grad = {demo.weight.grad.item():.4f}")

print("\nWITH zero_grad — same gradient every time, as it should be:")
for step in range(3):
    demo.zero_grad()
    loss = demo_loss_fn(demo(x), y)
    loss.backward()
    print(f"  step {step}: weight.grad = {demo.weight.grad.item():.4f}")

## 6. Your turn

A few one-line experiments that make the concepts stick:

- Set `lr=2.0` and rerun the loop — watch the loss blow up (overshooting).
- Set `lr=0.001` — watch it crawl and barely move in 100 epochs.
- Swap `torch.optim.SGD` for `torch.optim.Adam` and see how much faster it converges.
- Change the data rule to `y = -3x + 0.5` and confirm the model recovers the new numbers.

Remember to rerun the data + model cells before each experiment, so you start from a fresh random model.

---
Next up: **Part 3 — Datasets and DataLoaders**, how to feed a training loop without melting your machine.